# Module 1 — System Design for Multi-Agent Workflows

> **Format:** Paper-and-keyboard. No LLM calls — we're designing, not implementing.
>
> **Time:** 15 minutes.

By the end of this notebook you'll have:

1. A typed `TriageState` schema that defines what flows between agents
2. A graph topology — which nodes exist, which edges connect them
3. A short "anti-pattern check" exercise to catch design mistakes before they become bugs

Implementation comes in Module 2.

## Part A  —  The state schema

Every multi-agent system has **one shared state object** that every node reads and updates. Getting this right is half the design.

For our **Customer Support Triage Agent**, the state should hold:

- the original `ticket` text
- what the Classifier decided (`category`, `urgency`)
- what the Retriever pulled (`retrieved` policies)
- what the Drafter wrote (`draft`)
- what QA said (`verdict`, plus a `revision_count` for our termination guard)
- a history of `revisions` — useful in observability and eval (Module 4 + 5)

In [ ]:
from typing import TypedDict, Annotated, Literal
from operator import add


class TriageState(TypedDict):
    # TODO 1 — the raw ticket text from the customer
    ticket: ...

    # TODO 2 — what the Classifier decides
    # Hint: use Literal[...] for bounded values, "| None" for "not set yet"
    category: ...
    urgency:  ...

    # TODO 3 — what the Retriever pulled (list of dicts from the KB)
    retrieved: ...

    # TODO 4 — what the Drafter wrote
    draft: ...

    # TODO 5 — what QA decided
    verdict: ...

    # TODO 6 — termination guard counter
    revision_count: ...

    # TODO 7 — list that accumulates each draft attempt
    # Hint: this needs a reducer so nodes APPEND rather than REPLACE
    revisions: ...


# Sanity check — uncomment when you're done filling in the TODOs
# initial_state: TriageState = {
#     "ticket": "Why is my bill so high?",
#     "category": None,
#     "urgency": None,
#     "retrieved": [],
#     "draft": "",
#     "verdict": None,
#     "revision_count": 0,
#     "revisions": [],
# }
# print(initial_state)

**Hints**:

- `Literal["a", "b", "c"] | None` is great for category/urgency/verdict — bounded values, no typos.
- For the `revisions` list, use `Annotated[list[str], add]` so each node *appends* rather than *replaces*. This is the "reducer pattern" — Module 1's slides covered it.
- `revision_count` is a plain `int`; it starts at 0 and the QA node increments it.
- The `retrieved` list holds dicts from the KB, so type it as `list[dict]`.

## Part B  —  The graph topology

Module 1 slides 7–8 introduced four orchestration patterns. For the triage system, the design we converged on is:

**Sequential**, with **one feedback edge** from QA back to Drafter, and a hard `max_revisions = 2` guard.

Let's express it as Python data first — no LangGraph code yet. Just nodes, edges, and the routing logic.

In [ ]:
MAX_REVISIONS = 2

# TODO 1 — the four agent node names
NODES = {
    # "classify", ...
}

# TODO 2 — the static (always-fires) edges
STATIC_EDGES = [
    # ("classify", "retrieve"),
    # ...
]

# Entry point
ENTRY_POINT = "classify"

# TODO 3 — the conditional edge from QA
def route_qa(state) -> str:
    # If QA says pass, we're done
    # If QA says revise AND we haven't hit max_revisions, go back to drafter
    # If we hit max_revisions, force END (the termination guard)
    pass


# Sanity check — uncomment when you're done
# from pprint import pprint
# pprint({
#     "nodes": NODES,
#     "static_edges": STATIC_EDGES,
#     "entry_point": ENTRY_POINT,
#     "router_for_qa": route_qa.__name__,
# })

**Hints**:

- The `NODES` set is just the four agent names — strings.
- `STATIC_EDGES` are pairs that always fire in order: classify → retrieve → drafter → qa.
- `route_qa` is the conditional edge — it reads `state["verdict"]` and `state["revision_count"]` and returns the **next node name** (or `"END"` to stop).
- Don't forget the termination case: if `revision_count >= MAX_REVISIONS`, we go to `"END"` even if QA says revise. This is your guard against Module 1's "missing termination" anti-pattern.

## Part C  —  Anti-pattern check

Module 1 slide 10 named three design anti-patterns: **too many agents**, **missing termination**, **implicit state**.

For each scenario below, classify which anti-pattern is being committed (if any). Write your answer as a string in the dict below. Possible answers:
- `"too many agents"`
- `"missing termination"`
- `"implicit state"`
- `"no anti-pattern"`

In [ ]:
scenarios = [
    {
        "id": "S1",
        "description": (
            "A team splits their workflow into 9 micro-agents — one for "
            "category, one for sub-category, one for urgency, one for tone, "
            "one for politeness, one for greeting, etc. Each agent does ~5 "
            "lines of work."
        ),
        # TODO — which anti-pattern?
        "anti_pattern": "...",
    },
    {
        "id": "S2",
        "description": (
            "A Drafter agent that builds its reply by referencing 'the "
            "previous output' — but that output is never written into "
            "shared state. It just happens to be in scope from the last "
            "function call."
        ),
        # TODO
        "anti_pattern": "...",
    },
    {
        "id": "S3",
        "description": (
            "A feedback loop with no max_iterations check. Reviewer keeps "
            "saying 'try again' and the graph keeps looping. Cost goes "
            "from $0.02 per run to $4.10."
        ),
        # TODO
        "anti_pattern": "...",
    },
    {
        "id": "S4",
        "description": (
            "Three agents — Classifier, Retriever, Drafter — with clear "
            "typed state, a fixed sequential topology, and an explicit "
            "END after Drafter."
        ),
        # TODO
        "anti_pattern": "...",
    },
]

for s in scenarios:
    print(f"  {s['id']}:  {s['anti_pattern']}")

## Part D  —  A small design decision

Two teams pitch the same triage system with slightly different topologies. Pick one and write a one-sentence justification.

**Topology A**:  classify → retrieve → drafter → qa  (feedback to drafter, max 2 revisions)

**Topology B**:  classify → retrieve → drafter → tone_checker → fact_checker → policy_checker (no feedback loop, 3 separate reviewer agents in sequence)

Which would you ship? Why?

In [ ]:
# TODO — pick "A" or "B" and write a one-sentence reason
CHOICE = "..."
REASON = (
    "..."
)

print(f"I'd ship topology {CHOICE} because: {REASON}")

## Wrap up

That's Module 1. Three things to confirm before we move on:

1. **State** — every field is typed, no "maybe present" surprises.
2. **Topology** — every edge is either static or conditional with a clear router.
3. **Termination** — there's a hard ceiling on revisions; no infinite loops.

If those three are locked in, you're ready for **Module 2 — Building the End-to-End System with LangGraph**, where the design becomes runnable code.